# Originality

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch
import math
import pandas as pd

In [ ]:
MODEL_NAME = "openai-community/gpt2"
# m3hrdadfi/gpt2-persian-qa
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
@torch.no_grad()
def sentence_nll(sentence: str) -> float:
    # NLL per token (natural log)
    enc = tokenizer(sentence, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attn_mask = enc["attention_mask"].to(device)

    # Shift for LM training objective
    outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=input_ids)
    # loss is mean cross-entropy over tokens in batch (in nats)
    nll = outputs.loss.item() * input_ids.size(1)
    # average per token NLL
    avg_nll = outputs.loss.item()
    return avg_nll  # smaller = more likely, thus less novel

def sentence_perplexity(sentence: str) -> float:
    avg_nll = sentence_nll(sentence)
    ppl = math.exp(avg_nll)
    return ppl  # smaller = more fluent / common, less novel

def novelty_score(sentence: str, min_ppl=1.0, max_ppl=1000.0) -> float:
    """
    Map perplexity to [0,1] novelty.
    min_ppl: PPL at which novelty≈0
    max_ppl: PPL at which novelty≈1
    """
    ppl = sentence_perplexity(sentence)
    # clip
    ppl = max(min_ppl, min(max_ppl, ppl))
    # normalize
    score = (ppl - min_ppl) / (max_ppl - min_ppl)
    return score



In [ ]:
df = pd.read_csv('/content/majority_human_text.csv')

In [ ]:
sentences = df['Sentence']

In [ ]:
objective_originality = []
for s in sentences:
    n = novelty_score(s)
    objective_originality.append(n)



# Diversity

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import glob

In [ ]:
import torch

# model = SentenceTransformer('heydariAI/persian-embeddings')
# "xmanii/maux-gte-persian"
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', trust_remote_code=True)

if torch.cuda.is_available():
    model.to('cuda')
    print("Model moved to GPU.")
else:
    print("GPU not available, running on CPU.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model moved to GPU.


In [ ]:
df = pd.read_csv('/content/majority_human_all.csv')

In [ ]:
def calculate_similarity_diversity(sentences1:list, sentences2:list, model, k=0):
    """
    Calculates the average cosine similarity between the lower triangle
    of the similarity matrix of two lists of sentences.
    Similarity between each pair of sentences is computed using the
    SentenceTransformer model.

    Args:
        sentences1 (list): First list of sentences.
        sentences2 (list): Second list of sentences.
        model: SentenceTransformer model.
        k (int): Diagonal offset for `np.tril_indices_from`.
                 k=0 includes the main diagonal, k=-1 excludes it.

    Returns:
        float: Average similarity of the lower triangle.
    """
    # Ensure all sentences are strings
    sentences1 = [str(s) for s in sentences1 if s is not None]
    sentences2 = [str(s) for s in sentences2 if s is not None]

    if not sentences1 or not sentences2:
        raise ValueError("At least one of the sentence lists is empty.")
        # return np.nan # Return NaN if one of the sentence lists is empty

    embeddings1 = model.encode(sentences1, convert_to_numpy=True, show_progress_bar=False)
    embeddings2 = model.encode(sentences2, convert_to_numpy=True, show_progress_bar=False)

    # normalize embeddings
    normed_emb1 = embeddings1 / np.linalg.norm(embeddings1, axis=1, keepdims=True)
    normed_emb2 = embeddings2 / np.linalg.norm(embeddings2, axis=1, keepdims=True)

    # compute similarity matrix
    similarity_matrix = np.matmul(normed_emb1, normed_emb2.T)
    return similarity_matrix


In [ ]:
diversity = []
for i in range(5):
    sentences = df['Sentence'].iloc[20*i:20*(i+1)].tolist()
    sim_mat = calculate_similarity_diversity(sentences, sentences, model, k=0)

    # Calculate the average of each column in sim_mat, excluding the diagonal element
    num_sentences = sim_mat.shape[0]

    for j in range(num_sentences):
        col_values = sim_mat[:, j] # Get all values in column j
        sum_excluding_diagonal = col_values.sum() - sim_mat[j, j]
        average_excluding_diagonal = sum_excluding_diagonal / (num_sentences - 1)
        diversity.append(average_excluding_diagonal)

    print(f"Topic {i+1} - Average column similarities (excluding diagonal): {diversity}")

Topic 1 - Average column similarities (excluding diagonal): [np.float32(0.5597087), np.float32(0.5745908), np.float32(0.5710483), np.float32(0.56463224), np.float32(0.5908643), np.float32(0.59218156), np.float32(0.5253988), np.float32(0.59669983), np.float32(0.6166444), np.float32(0.57491976), np.float32(0.5655185), np.float32(0.5339414), np.float32(0.58733386), np.float32(0.51831937), np.float32(0.59444064), np.float32(0.4602242), np.float32(0.55862635), np.float32(0.5694256), np.float32(0.5278468), np.float32(0.57794636)]
Topic 2 - Average column similarities (excluding diagonal): [np.float32(0.5597087), np.float32(0.5745908), np.float32(0.5710483), np.float32(0.56463224), np.float32(0.5908643), np.float32(0.59218156), np.float32(0.5253988), np.float32(0.59669983), np.float32(0.6166444), np.float32(0.57491976), np.float32(0.5655185), np.float32(0.5339414), np.float32(0.58733386), np.float32(0.51831937), np.float32(0.59444064), np.float32(0.4602242), np.float32(0.55862635), np.float32

In [ ]:
df['Diversity'] = diversity

# Quality

In [ ]:
import random

def corrupt_sentence(sentence: str, prob_shuffle=0.3, prob_drop=0.2):
    tokens = sentence.split()
    # random drop
    tokens = [t for t in tokens if random.random() > prob_drop or len(tokens) <= 3]
    # random small shuffle
    if random.random() < prob_shuffle and len(tokens) > 3:
        i, j = sorted(random.sample(range(len(tokens)), 2))
        tokens[i], tokens[j] = tokens[j], tokens[i]
    return " ".join(tokens)

def fluency_score_contrastive(sentence: str, n_corrupt=5):
    orig_ppl = sentence_perplexity(sentence)
    cor_ppls = []
    for _ in range(n_corrupt):
        corrupted = corrupt_sentence(sentence)
        if corrupted.strip():
            cor_ppls.append(sentence_perplexity(corrupted))
    if not cor_ppls:
        return 0.5
    avg_cor_ppl = sum(cor_ppls) / len(cor_ppls)
    # if original << corrupted, sentence is fluent
    ratio = avg_cor_ppl / (orig_ppl + 1e-8)
    # map ratio to [0,1] with a saturating function
    # ratio=1 -> 0.5, ratio=2 -> ~0.73, ratio=4 -> ~0.89, etc.
    score = 1.0 / (1.0 + (1.0 / max(ratio, 1e-6)))
    return max(0.0, min(1.0, score))


In [ ]:
objective_quality = []
for s in sentences:
    n = fluency_embedding_length_aware(s)
    objective_quality.append(n)

# fluency_embedding_length_aware

In [ ]:
final_df['Model centroid similarity'] = objective_quality